In [ ]:
from pathlib import Path

import polars as pl
import matplotlib.pyplot as plt

In [ ]:
KAGGLE_INPUT_BASE_PATH = "/kaggle/input/m5-forecasting-accuracy"
LOCAL_INPUT_BASE_PATH = "./data"
INPUT_BASE_PATH = LOCAL_INPUT_BASE_PATH

EDA_RESULTS_PATH = Path("./results/eda/")
EDA_RESULTS_PATH.mkdir(parents=True, exist_ok=True)

CALENDAR_DATA = pl.read_csv(f"{INPUT_BASE_PATH}/calendar.csv", try_parse_dates=True)
SELL_PRICES = pl.read_csv(f"{INPUT_BASE_PATH}/sell_prices.csv")
SALES_TRAIN_EVALUATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_evaluation.csv")
SALES_TRAIN_VALIDATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_validation.csv")
SAMPLE_SUBMISSION = pl.read_csv(f"{INPUT_BASE_PATH}/sample_submission.csv")

In [ ]:
EDA_RESULTS_PATH = Path("./results/eda/")
EDA_RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Data preprocessing

In [ ]:
full_df = (
    SALES_TRAIN_EVALUATION.unpivot(
        index=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
        variable_name="d",
        value_name="sales",
    )
    .join(
        CALENDAR_DATA.select(pl.all().exclude("weekday", "wday", "month", "year")),
        on="d",
        how="left",
    )
    .join(
        SELL_PRICES,
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left",
    )
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.UInt64))
)

full_df.head()

In [ ]:
# Plot sales timeseries for each store and department.

store_ids = full_df["store_id"].unique().sort().to_list()
dept_ids = full_df["dept_id"].unique().sort().to_list()

for store_id in store_ids:
    fig, axes = plt.subplots(len(dept_ids), 1, figsize=(10, len(dept_ids) * 2), sharex=True)
    for i, dept_id in enumerate(dept_ids):
        store_dept_df = (
            full_df
            .filter(pl.col("store_id") == store_id, pl.col("dept_id") == dept_id)
            .select(pl.col("item_id"), pl.col("d_index"), pl.col("sales"))

        )
        
        item_ids = store_dept_df["item_id"].unique().sort().to_list()
        for item_id in item_ids:
            item_df = (
                store_dept_df
                .filter(pl.col("item_id") == item_id)
                .sort(by=pl.col("d_index"))
            )
            axes[i].plot(
                item_df["d_index"].to_numpy(),
                item_df["sales"].to_numpy(),
                lw=0.25,
                color="grey",
                alpha=0.25,
            )

        avg_sales_df = (
            store_dept_df
            .group_by("d_index")
            .agg(
                pl.col("sales").mean().name.suffix("_mean"),
                pl.col("sales").quantile(0.5).name.suffix("_median"),
                pl.col("sales").quantile(0.95).name.suffix("_q95"),
                pl.col("sales").quantile(0.05).name.suffix("_q05")
            )
            .sort("d_index")
        )
        axes[i].plot(
            avg_sales_df["d_index"].to_numpy(),
            avg_sales_df["sales_mean"].to_numpy(),
            color="C0",
            lw=0.75,
        )
        
        axes[i].set(yscale="log", ylabel=dept_id)
        if i == len(dept_ids) - 1:
            axes[i].set(xlabel="Timestamp Index")

    fig.align_labels()
    fig.suptitle(store_id)
    fig.tight_layout();

    plt.savefig(str(EDA_RESULTS_PATH / f"sales_timeseries_by_dept_store_{store_id}.png"), dpi=300);
    plt.close()

In [ ]:
# Average sales by department for single store

store_ids = full_df["store_id"].unique().sort().to_list()
dept_ids = full_df["dept_id"].unique().sort().to_list()

for store_id in store_ids:
    fig, axes = plt.subplots(len(dept_ids), 1, figsize=(12.5, len(dept_ids) * 1.5), sharex=True)
    for i, dept_id in enumerate(dept_ids):
        store_dept_df = (
            full_df
            .filter(pl.col("store_id") == store_id, pl.col("dept_id") == dept_id)
            .select(pl.col("item_id"), pl.col("d_index"), pl.col("sales"))
        )
        avg_sales_df = (
            store_dept_df
            .group_by("d_index")
            .agg(
                pl.col("sales").mean().name.suffix("_mean"),
                pl.col("sales").quantile(0.5).name.suffix("_median"),
                pl.col("sales").quantile(0.95).name.suffix("_q95"),
                pl.col("sales").quantile(0.05).name.suffix("_q05")
            )
            .sort("d_index")
        )
        axes[i].plot(
            avg_sales_df["d_index"].to_numpy(),
            avg_sales_df["sales_mean"].to_numpy(),
            color="C0",
            lw=0.75,
        )
        axes[i].grid(ls="--", alpha=0.5)
        axes[i].set(ylabel=dept_id)
        if i == len(dept_ids) - 1:
            axes[i].set(xlabel="Timestamp Index")

    fig.align_labels()
    fig.suptitle(store_id)
    fig.tight_layout();

    plt.savefig(str(EDA_RESULTS_PATH / f"avg_dept_sales_timeseries_store_{store_id}.png"), dpi=300);
    plt.close()

## Price data analysis

In [ ]:
no_price_df = full_df.filter(pl.col("sell_price").is_null())


# Are periods of missing prices single continuous periods?
# Or are the multiple periods over which there are missing prices?
items_ids_no_price = no_price_df["item_id"].unique().sort().to_list()

item_ids_with_non_cont_missing_prices = []
for item_id in items_ids_no_price:
    item_df = (
        no_price_df
        .filter(pl.col("item_id") == item_id)
        .sort(by=pl.col("d_index"))
        .select(pl.col("d_index").diff().alias("d_index_diff"))
        .filter((~pl.col("d_index_diff").is_null()), pl.col("d_index_diff").gt(1))
    )
    if not item_df.is_empty():
        item_ids_with_non_cont_missing_prices.append(item_id)


len(item_ids_with_non_cont_missing_prices)

In [ ]:
# Do we ever sell anything when price is null?
full_df.filter(pl.col("sell_price").is_null(), pl.col("sales").gt(0))

In [ ]:
# Inspect timeseries of sales for when prices are null.

items_ids_no_price = no_price_df["id"].unique().sort().to_list()

start_idx = 468
n_items_to_plot = 10
item_ids_to_plot = items_ids_no_price[int(start_idx * n_items_to_plot): int((start_idx + 1) * n_items_to_plot)]

fig, axes = plt.subplots(len(item_ids_to_plot), 1, figsize=(10, len(item_ids_to_plot) * 1.5), sharex=True)
for i, item_id in enumerate(item_ids_to_plot):
    no_price_item_df = (
        no_price_df
        .filter(pl.col("id") == item_id)
        .select(pl.col("id"), pl.col("item_id"), pl.col("d_index"), pl.col("sales"))
        .sort(by=pl.col("d_index"))
    )
    axes[i].scatter(
        no_price_item_df["d_index"].to_numpy(),
        no_price_item_df["sales"].to_numpy(),
        marker="x",
        s=5,
        color="tab:red",
        alpha=0.25,
    )
    
    full_item_df = (
        full_df
        .filter(pl.col("id") == item_id)
        .select(pl.col("id"), pl.col("item_id"), pl.col("d_index"), pl.col("sales"))
        .sort(by=pl.col("d_index"))
    )
    axes[i].plot(
        full_item_df["d_index"].to_numpy(),
        full_item_df["sales"].to_numpy(),
        lw=0.75,
        color="grey",
        alpha=0.75,
        label=item_id.rstrip("_evaluation")
    )

    axes[i].legend(loc=1)
    axes[i].set(ylabel="sales")
    if i == len(item_ids_to_plot) - 1: axes[i].set(xlabel="timestamp index")

fig.align_labels()
fig.tight_layout();

# plt.savefig(str(EDA_RESULTS_PATH / f"sales_timeseries_by_dept_store_{store_id}.png"), dpi=300);
# plt.close()

In [ ]:
# Check that for each item without sell price, the period without prices
# is at the start of the sales timeseries.

items_ids_no_price = no_price_df["id"].unique().sort().to_list()
for item_id in items_ids_no_price:
    no_price_item_df = (
        no_price_df
        .filter(pl.col("id") == item_id)
        .select(pl.col("id"), pl.col("item_id"), pl.col("d_index"), pl.col("sales"), pl.col("sell_price"))
        .sort(by=pl.col("d_index"))
    )
    assert no_price_item_df["d_index"].min() == 1
    assert no_price_item_df["d_index"].max() == len(no_price_item_df)
    assert (no_price_item_df["d_index"] == pl.arange(1, len(no_price_item_df) + 1, eager=True, dtype=pl.UInt64).alias("d_index")).all()


## Forecast Period Analysis